# Laboratorio — Optimización numérica

**Diplomado en Aprendizaje Máquina y Modelos Generativos · Módulo III**

El laboratorio conecta los capítulos del módulo mediante cuatro preguntas:

1. ¿Cómo se ve la geometría de una función que queremos minimizar?
2. ¿Cómo cambia el comportamiento al intercambiar el optimizador?
3. ¿Qué significa realmente que un método converja más rápido?
4. ¿Cómo se relacionan iteraciones, evaluaciones de gradiente y tiempo computacional?

Los algoritmos se mantienen separados en `optimizadores.py`. Este notebook se concentra en **experimentar, medir, comparar e interpretar**.

> **Idea clave:** menos iteraciones no implica necesariamente menor costo computacional.

## 0. Preparación

El notebook espera que `optimizadores.py` esté en el mismo directorio. Las implementaciones usan una interfaz común y una fábrica, de modo que el pipeline puede cambiar de algoritmo sin cambiar el problema experimental.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from optimizadores import (
    OptimizationProblem,
    NumericalOptimizerFactory,
    OptimizationPipeline,
)

np.set_printoptions(precision=5, suppress=True)
plt.rcParams["figure.figsize"] = (7, 5)

SEED = 42
factory = NumericalOptimizerFactory()
print("Optimizadores disponibles:", factory.available())

## 1. Funciones de prueba: el paisaje antes del algoritmo

Las funciones de prueba permiten aislar dificultades geométricas antes de introducir un dataset real. No todas representan una pérdida típica de aprendizaje máquina; su utilidad es mostrar propiedades que desafían a los algoritmos.

Trabajaremos visualmente con **parábola, esfera, Rosenbrock, Rastrigin y Ackley**.

- **Parábola:** caso elemental convexo.
- **Esfera:** convexa, suave e isotrópica.
- **Rosenbrock:** valle curvo y estrecho.
- **Rastrigin:** altamente multimodal.
- **Ackley:** multimodal, con regiones relativamente planas y estructura oscilatoria.

Las curvas de nivel serán útiles después para interpretar trayectorias de optimización.

In [ ]:
def parabola(x):
    return x**2

def sphere(theta):
    theta = np.asarray(theta)
    return np.sum(theta**2, axis=-1)

def rosenbrock(theta, a=1.0, b=100.0):
    theta = np.asarray(theta)
    x, y = theta[..., 0], theta[..., 1]
    return (a - x)**2 + b * (y - x**2)**2

def rastrigin(theta, A=10.0):
    theta = np.asarray(theta)
    d = theta.shape[-1]
    return A*d + np.sum(theta**2 - A*np.cos(2*np.pi*theta), axis=-1)

def ackley(theta, a=20.0, b=0.2, c=2*np.pi):
    theta = np.asarray(theta)
    d = theta.shape[-1]
    s1 = np.sum(theta**2, axis=-1)
    s2 = np.sum(np.cos(c*theta), axis=-1)
    return -a*np.exp(-b*np.sqrt(s1/d)) - np.exp(s2/d) + a + np.e

In [ ]:
x = np.linspace(-4, 4, 400)
fig, ax = plt.subplots()
ax.plot(x, parabola(x))
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Parábola: f(x) = x²")
ax.grid(alpha=0.25)
plt.show()

In [ ]:
def plot_test_function(function, xlim, ylim, title, levels=35, n=250):
    x = np.linspace(*xlim, n)
    y = np.linspace(*ylim, n)
    X, Y = np.meshgrid(x, y)
    Z = function(np.stack([X, Y], axis=-1))

    fig, ax = plt.subplots()
    contour = ax.contourf(X, Y, Z, levels=levels)
    ax.contour(X, Y, Z, levels=levels, linewidths=0.35)
    ax.set_xlabel("θ₁")
    ax.set_ylabel("θ₂")
    ax.set_title(title)
    fig.colorbar(contour, ax=ax, label="f(θ)")
    plt.show()

plot_test_function(sphere, (-3, 3), (-3, 3), "Esfera")
plot_test_function(rosenbrock, (-2, 2), (-1, 3), "Rosenbrock", levels=45)
plot_test_function(rastrigin, (-5.12, 5.12), (-5.12, 5.12), "Rastrigin", levels=45)
plot_test_function(ackley, (-5, 5), (-5, 5), "Ackley", levels=45)

### Predice antes de ejecutar

Observa las geometrías y formula una predicción:

- ¿En cuál esperarías que la dirección del gradiente sea especialmente informativa?
- ¿Cuál puede producir trayectorias oscilatorias?
- ¿En cuáles un mínimo local no caracteriza el problema global?
- ¿Qué dificultad parece provenir de **curvatura** y cuál de **multimodalidad**?

## 2. Problema controlado

Consideremos la función cuadrática

\[
f(\theta)=\frac{1}{2}\theta^\top A\theta-b^\top\theta,
\]

con \(A\) simétrica definida positiva. Entonces

\[
\nabla f(\theta)=A\theta-b,\qquad H(\theta)=A,\qquad
\theta^\star=A^{-1}b.
\]

Esto permite conocer el óptimo y controlar el condicionamiento.

In [ ]:
def make_quadratic(A, b):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    def f(theta):
        return 0.5 * theta @ A @ theta - b @ theta

    def grad(theta):
        return A @ theta - b

    def hess(theta):
        return A

    return OptimizationProblem(f, grad, hess)

A = np.array([[8.0, 3.0], [3.0, 2.0]])
b = np.array([1.0, 1.0])
quadratic = make_quadratic(A, b)

theta_star = np.linalg.solve(A, b)
theta0 = np.array([-2.0, 2.0])

print("theta* =", theta_star)
print("f(theta*) =", quadratic.f(theta_star))
print("cond(A) =", np.linalg.cond(A))

## 3. Pipeline: cambiar el algoritmo, no el experimento

La fábrica crea la estrategia concreta. `OptimizationPipeline` conoce solamente la abstracción del optimizador.

In [ ]:
gd = factory.create("gd", learning_rate=0.08, max_iter=250, tol=1e-8)
pipeline = OptimizationPipeline(quadratic, gd)
res_gd = pipeline.run(theta0)

newton = factory.create("newton", max_iter=20, tol=1e-8)
pipeline.set_optimizer(newton)
res_newton = pipeline.run(theta0)

bfgs = factory.create("bfgs", max_iter=100, tol=1e-8)
pipeline.set_optimizer(bfgs)
res_bfgs = pipeline.run(theta0)

dfp = factory.create("dfp", max_iter=100, tol=1e-8)
pipeline.set_optimizer(dfp)
res_dfp = pipeline.run(theta0)

results_quadratic = {
    "GD": res_gd, "Newton": res_newton,
    "BFGS": res_bfgs, "DFP": res_dfp
}

for name, result in results_quadratic.items():
    print(f"{name:8s} | k={result.iterations:3d} | "
          f"f={quadratic.f(result.theta): .4e} | "
          f"t={result.elapsed_time:.4e}s | {result.stop_reason}")

## 4. Trayectorias de optimización

En dos dimensiones podemos observar **cómo** llega cada método a la solución. Una trayectoria corta en iteraciones puede esconder operaciones más costosas.

In [ ]:
def plot_trajectories(problem, results, optimum,
                      xlim=(-2.5, 1.0), ylim=(-0.5, 2.5)):
    x = np.linspace(*xlim, 220)
    y = np.linspace(*ylim, 220)
    X, Y = np.meshgrid(x, y)
    Z = np.array([
        [problem.f(np.array([xx, yy])) for xx in x]
        for yy in y
    ])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.contour(X, Y, Z, levels=30)

    for name, result in results.items():
        trajectory = np.asarray(result.historial["theta"])
        ax.plot(trajectory[:, 0], trajectory[:, 1],
                marker="o", markersize=3, linewidth=1.5, label=name)

    ax.scatter(optimum[0], optimum[1], marker="*", s=160, label="θ*")
    ax.set_xlabel("θ₁")
    ax.set_ylabel("θ₂")
    ax.set_title("Trayectorias sobre la función cuadrática")
    ax.legend()
    plt.show()

plot_trajectories(quadratic, results_quadratic, theta_star)

## 5. ¿Qué significa converger más rápido?

Distinguiremos

\[
\text{iteraciones},\qquad
\text{trabajo computacional},\qquad
\text{tiempo}.
\]

Cuando conocemos \(f^\star\), estudiaremos \(f(\theta_k)-f^\star\). Cuando el óptimo no es conocido, utilizaremos pérdida y norma del gradiente como diagnósticos complementarios.

In [ ]:
f_star = quadratic.f(theta_star)

fig, ax = plt.subplots()
for name, result in results_quadratic.items():
    h = result.historial
    gap = np.maximum(np.asarray(h["perdida"]) - f_star, 1e-16)
    ax.semilogy(h["iteracion"], gap, label=name)

ax.set_xlabel("Iteración")
ax.set_ylabel("f(θₖ) - f*")
ax.set_title("Convergencia medida por iteraciones")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots()
for name, result in results_quadratic.items():
    h = result.historial
    gap = np.maximum(np.asarray(h["perdida"]) - f_star, 1e-16)
    ax.semilogy(h["tiempo"], gap, label=name)

ax.set_xlabel("Tiempo acumulado [s]")
ax.set_ylabel("f(θₖ) - f*")
ax.set_title("Convergencia medida por tiempo")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 6. Complejidad algorítmica

Sea \(d\) el número de parámetros, \(n\) el número de observaciones y \(B\) el tamaño de lote.

Para un modelo cuya contribución por observación cuesta \(O(d)\), un gradiente con lote \(B\) tiene típicamente costo proporcional a \(O(Bd)\). En full batch, \(B=n\).

Newton introduce la Hessiana, almacenamiento \(O(d^2)\), y la resolución de un sistema lineal denso puede costar \(O(d^3)\). DFP y BFGS evitan calcular la Hessiana exacta, pero una implementación densa de la aproximación inversa requiere almacenamiento y actualizaciones de orden cuadrático en \(d\).

Estas expresiones son modelos asintóticos, no cronómetros.

In [ ]:
complexity = pd.DataFrame([
    ["GD / SGD / Momentum", "O(Bd)", "O(d)", "Gradiente"],
    ["AdaGrad / RMSProp / Adam", "O(Bd)", "O(d)", "Gradiente + estados"],
    ["Newton (denso)", "gradiente + Hessiana + O(d³)", "O(d²)", "Sistema lineal"],
    ["DFP / BFGS", "gradiente + O(d²)", "O(d²)", "Actualización matricial"],
], columns=["Familia", "Costo orientativo por actualización",
            "Memoria adicional", "Operación dominante"])
complexity

## 7. Problema basado en datos: regresión logística

Generaremos un problema sintético de clasificación binaria para que \(n\), \(d\) y \(B\) tengan significado explícito. Usaremos pérdida logística promedio con regularización L2; el sesgo no se regulariza.

In [ ]:
def make_binary_classification(n=2000, d=8, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, d))
    true_w = rng.normal(size=d)
    logits = X @ true_w + 0.25 * rng.normal(size=n)
    y = (logits > 0).astype(float)
    return np.column_stack([np.ones(n), X]), y

X, y = make_binary_classification(n=2500, d=10, seed=SEED)

def make_logistic_problem(X, y, reg=1e-3):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape
    mask = np.ones(d)
    mask[0] = 0.0

    def select(indices):
        return (X, y) if indices is None else (X[indices], y[indices])

    def sigmoid(z):
        out = np.empty_like(z)
        pos = z >= 0
        out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
        ez = np.exp(z[~pos])
        out[~pos] = ez / (1.0 + ez)
        return out

    def objective(theta, indices=None):
        Xb, yb = select(indices)
        z = Xb @ theta
        return (np.mean(np.logaddexp(0.0, z) - yb*z)
                + 0.5*reg*np.sum((mask*theta)**2))

    def gradient(theta, indices=None):
        Xb, yb = select(indices)
        p = sigmoid(Xb @ theta)
        return Xb.T @ (p-yb)/len(yb) + reg*mask*theta

    def hessian(theta, indices=None):
        Xb, _ = select(indices)
        p = sigmoid(Xb @ theta)
        w = p*(1-p)
        return (Xb.T*w) @ Xb/len(Xb) + reg*np.diag(mask)

    return OptimizationProblem(objective, gradient, hessian, n_samples=n)

logistic = make_logistic_problem(X, y)
theta0_log = np.zeros(X.shape[1])
print("n =", X.shape[0], "| d =", X.shape[1])

## 8. Tamaño de lote: SGD → mini-batch → full batch

Compararemos \(B=1\), dos tamaños de mini-batch y full batch. La pregunta central es cuánto trabajo representa cada actualización.

In [ ]:
batch_experiments = {
    "SGD B=1": factory.create(
        "sgd", learning_rate=0.03, max_iter=500, tol=1e-5, seed=SEED),
    "Mini-batch B=32": factory.create(
        "gd", learning_rate=0.08, batch_size=32,
        max_iter=500, tol=1e-5, seed=SEED),
    "Mini-batch B=256": factory.create(
        "gd", learning_rate=0.12, batch_size=256,
        max_iter=500, tol=1e-5, seed=SEED),
    "Full batch": factory.create(
        "gd", learning_rate=0.25, batch_size=None,
        max_iter=250, tol=1e-5, seed=SEED),
}

batch_results = {}
pipeline = OptimizationPipeline(logistic, batch_experiments["SGD B=1"])

for name, optimizer in batch_experiments.items():
    pipeline.set_optimizer(optimizer)
    batch_results[name] = pipeline.run(theta0_log)
    r = batch_results[name]
    print(f"{name:18s} | k={r.iterations:4d} | "
          f"loss={r.historial['perdida'][-1]:.6f} | "
          f"t={r.elapsed_time:.4f}s")

In [ ]:
def plot_metric(results, xkey, xlabel, title):
    fig, ax = plt.subplots()
    for name, result in results.items():
        h = result.historial
        ax.plot(h[xkey], h["perdida"], label=name)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Pérdida")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()

plot_metric(batch_results, "iteracion", "Actualización / iteración",
            "Pérdida vs. iteraciones")
plot_metric(batch_results, "gradientes_individuales",
            "Gradientes individuales evaluados",
            "Pérdida vs. trabajo en datos")
plot_metric(batch_results, "tiempo", "Tiempo acumulado [s]",
            "Pérdida vs. tiempo")

### Discusión

1. ¿Cambiarías tu conclusión si observas iteraciones en lugar de gradientes individuales?
2. ¿La conclusión cambia al observar tiempo?
3. ¿Dónde aparece el ruido de los lotes pequeños?
4. ¿Qué cantidad usarías para comparar algoritmos independientemente del hardware?

## 9. Comparación de optimizadores de primer orden

Mantendremos fijo el problema y compararemos distintas reglas de actualización. Los hiperparámetros son valores de demostración, no configuraciones universalmente óptimas.

In [ ]:
first_order = {
    "GD": factory.create("gd", learning_rate=0.25, max_iter=200, tol=1e-5, seed=SEED),
    "Momentum": factory.create("momentum", learning_rate=0.08, beta=0.9,
                               max_iter=200, tol=1e-5, seed=SEED),
    "AdaGrad": factory.create("adagrad", learning_rate=0.15,
                              max_iter=200, tol=1e-5, seed=SEED),
    "RMSProp": factory.create("rmsprop", learning_rate=0.02, beta=0.9,
                              max_iter=200, tol=1e-5, seed=SEED),
    "Adam": factory.create("adam", learning_rate=0.03, beta1=0.9, beta2=0.999,
                           max_iter=200, tol=1e-5, seed=SEED),
}

first_order_results = {}
pipeline = OptimizationPipeline(logistic, first_order["GD"])
for name, optimizer in first_order.items():
    pipeline.set_optimizer(optimizer)
    first_order_results[name] = pipeline.run(theta0_log)

fig, ax = plt.subplots()
for name, result in first_order_results.items():
    ax.semilogy(result.historial["iteracion"],
                result.historial["norma_gradiente"], label=name)
ax.set_xlabel("Iteración")
ax.set_ylabel("||∇f(θₖ)||")
ax.set_title("Primer orden: norma del gradiente")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 10. Tabla experimental común

La tabla no pretende producir un “ganador”. Conserva simultáneamente progreso, trabajo y tiempo para identificar compromisos.

In [ ]:
def summarize_results(results):
    rows = []
    for name, r in results.items():
        h = r.historial
        rows.append({
            "Método": name,
            "Convergió": r.converged,
            "Motivo": r.stop_reason,
            "Iteraciones": r.iterations,
            "Pérdida final": h["perdida"][-1],
            "||grad|| final": h["norma_gradiente"][-1],
            "Eval. gradiente": h["eval_gradiente"][-1],
            "Gradientes individuales": h["gradientes_individuales"][-1],
            "Eval. Hessiana": h["eval_hessiana"][-1],
            "Tiempo [s]": r.elapsed_time,
        })
    return pd.DataFrame(rows)

summarize_results(first_order_results)

## 11. Benchmark de tiempo

Una medición aislada puede estar contaminada por calentamiento y carga del sistema. Repetiremos cada ejecución y reportaremos mediana y rango.

Los tiempos son propios de **este entorno y estas implementaciones**; no deben confundirse con complejidad asintótica.

In [ ]:
def benchmark(problem, theta0, optimizer_builder, repeats=5):
    times, iterations, losses = [], [], []
    for _ in range(repeats):
        result = OptimizationPipeline(
            problem, optimizer_builder()
        ).run(theta0)
        times.append(result.elapsed_time)
        iterations.append(result.iterations)
        losses.append(result.historial["perdida"][-1])

    return {
        "mediana_s": np.median(times),
        "min_s": np.min(times),
        "max_s": np.max(times),
        "iteraciones_mediana": np.median(iterations),
        "perdida_mediana": np.median(losses),
    }

bench = {
    "GD": benchmark(logistic, theta0_log,
        lambda: factory.create("gd", learning_rate=0.25,
                               max_iter=150, tol=1e-5, seed=SEED)),
    "Adam": benchmark(logistic, theta0_log,
        lambda: factory.create("adam", learning_rate=0.03,
                               max_iter=150, tol=1e-5, seed=SEED)),
}
pd.DataFrame(bench).T

## 12. Escalabilidad

Variaremos \(n\) manteniendo \(d\) aproximadamente fijo. Este experimento **no demuestra** una cota \(O(\cdot)\); permite contrastar si la tendencia medida es consistente con las operaciones realizadas.

In [ ]:
def scaling_experiment(sample_sizes=(500, 1000, 2000, 4000), d=20):
    rows = []
    for n in sample_sizes:
        Xn, yn = make_binary_classification(n=n, d=d, seed=SEED)
        problem_n = make_logistic_problem(Xn, yn)
        theta0_n = np.zeros(Xn.shape[1])
        optimizer = factory.create(
            "gd", learning_rate=0.2, max_iter=50, tol=0.0, seed=SEED)
        result = OptimizationPipeline(problem_n, optimizer).run(theta0_n)
        rows.append({
            "n": n,
            "d": Xn.shape[1],
            "tiempo_s": result.elapsed_time,
            "gradientes_individuales":
                result.historial["gradientes_individuales"][-1],
        })
    return pd.DataFrame(rows)

scaling_n = scaling_experiment()
display(scaling_n)

fig, ax = plt.subplots()
ax.plot(scaling_n["n"], scaling_n["tiempo_s"], marker="o")
ax.set_xlabel("Número de observaciones n")
ax.set_ylabel("Tiempo [s]")
ax.set_title("Escalabilidad empírica con n")
ax.grid(alpha=0.25)
plt.show()

## 13. Actividad por equipos: implementar, no solamente ejecutar

Cada equipo seleccionará **un algoritmo estudiado en el módulo** e intentará implementarlo.

La implementación de referencia de `optimizadores.py` se utiliza en este notebook para la demostración. En la actividad, la meta es reconstruir el algoritmo a partir de su formulación matemática y comprobar experimentalmente su comportamiento.

### Entregable sugerido

1. Identificar la regla de actualización y variables de estado.
2. Implementar una clase compatible con la interfaz `Optimizer`.
3. Verificarla sobre una función de prueba sencilla.
4. Integrarla al mismo `OptimizationPipeline`.
5. Ejecutarla sobre el problema de regresión logística.
6. Reportar convergencia mediante pérdida y/o norma del gradiente.
7. Medir tiempo y trabajo computacional.
8. Explicar el costo esperado por iteración en función de \(n\), \(d\) y, cuando corresponda, \(B\).
9. Comparar sus observaciones con al menos una implementación de referencia.
10. Interpretar qué propiedades del algoritmo explican los resultados.

In [ ]:
# Esqueleto conceptual para la actividad:
#
# from optimizadores import Optimizer
#
# class MiOptimizador(Optimizer):
#     def optimize(self, problem, theta0):
#         # 1. Inicializar theta y variables de estado.
#         # 2. Evaluar la información necesaria.
#         # 3. Aplicar la regla de actualización.
#         # 4. Registrar métricas.
#         # 5. Verificar criterio de paro.
#         # 6. Devolver OptimizationResult.
#         raise NotImplementedError

## 14. Cierre

El laboratorio no busca establecer un optimizador universalmente superior. Busca aprender a **formular una comparación**.

Al interpretar un resultado, pregunta:

- ¿Qué geometría tiene el problema?
- ¿Qué información utiliza cada actualización?
- ¿Cuánto cuesta obtener esa información?
- ¿Qué estamos contando como una iteración?
- ¿Cuántos datos fueron procesados?
- ¿Qué criterio usamos para declarar convergencia?
- ¿El tiempo medido es consistente y reproducible?
- ¿La conclusión se mantiene al cambiar la métrica del eje horizontal?

La elección de un método involucra compromisos entre **información, costo, memoria, estabilidad y progreso hacia la solución**.

## Referencias

- Nocedal, J. & Wright, S. J. (2006). *Numerical Optimization*, 2nd ed. Springer.
- Bottou, L., Curtis, F. E. & Nocedal, J. (2018). “Optimization Methods for Large-Scale Machine Learning”. *SIAM Review*, 60(2), 223–311.
- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. MIT Press.
- Surjanovic, S. & Bingham, D. *Virtual Library of Simulation Experiments: Test Functions and Datasets*. Referencia complementaria para funciones de prueba.